# 04.1 — Dedicated digit classifier (MNIST pretrain -> fine-tune)

A small CNN that classifies the **comb digit cells** (pre-segmented single
digits) — the case where EasyOCR's sequence recognizer is weakest. Staged
**experiment**: run it, see if the per-digit accuracy and downstream exact-match
look good; if so we wire it into the main pipeline (digits -> this CNN, the 3
insurance letters + free text -> EasyOCR).

Pipeline: pretrain on MNIST -> fine-tune on this form's own digit cells (80 train
forms) -> evaluate per-digit accuracy and SSN/phone/date exact-match on the 20
held-out forms. Logic lives in `src/digit_model.py`.

Run on the PyTorch+CUDA server (GPU auto-detected). Needs `torchvision` (already
in the image) for MNIST.  Saves `models/digit_cnn/digit_cnn.pth`.

In [ ]:
import sys, random
from pathlib import Path
import numpy as np, pandas as pd, torch
from torch.utils.data import DataLoader

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
import ocr, finetune, digit_model as dm

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)

CROPS     = ROOT / "data" / "scans" / "crops"
FIELD_MAP = ocr.load_field_map(str(ROOT / "data" / "template" / "field_map.json"))
labels    = pd.read_csv(ROOT / "data" / "scans" / "labels.csv", dtype=str).fillna("")
labels_by_pid = {r["patient_id"]: dict(r) for _, r in labels.iterrows()}

SPLIT = ROOT / "data" / "generated" / "split_train_val.json"
if SPLIT.exists():
    train_pids, val_pids = finetune.load_split(str(SPLIT))
else:
    train_pids, val_pids = finetune.split_pids(sorted(labels_by_pid), 0.8, 42)
print(f"train {len(train_pids)} / test(held-out) {len(val_pids)} forms")

MNIST_DIR = ROOT / "data" / "mnist"
CKPT_DIR  = ROOT / "models" / "digit_cnn"; CKPT_DIR.mkdir(parents=True, exist_ok=True)
PRE  = CKPT_DIR / "mnist_pretrained.pth"
CKPT = CKPT_DIR / "digit_cnn.pth"

In [ ]:
# Collect digit cells (insurance's 3 letter cells are excluded automatically).
train_cells = dm.collect_digit_cells(str(CROPS), FIELD_MAP, labels_by_pid, train_pids)
test_cells  = dm.collect_digit_cells(str(CROPS), FIELD_MAP, labels_by_pid, val_pids)
print(f"train digit cells: {len(train_cells)}  |  test digit cells: {len(test_cells)}")
print("per-class (train):", dm.digit_histogram(train_cells))

## 1. Pretrain on MNIST

Then check how MNIST-only does on our form digits — that gap shows why the fine-tune step is needed.

In [ ]:
import torchvision as tv, torchvision.transforms as T
tf = T.Compose([T.ToTensor(), T.Normalize((0.1307,), (0.3081,))])
mn_tr = tv.datasets.MNIST(str(MNIST_DIR), train=True,  download=True, transform=tf)
mn_te = tv.datasets.MNIST(str(MNIST_DIR), train=False, download=True, transform=tf)
mn_tr_loader = DataLoader(mn_tr, batch_size=128, shuffle=True)
mn_te_loader = DataLoader(mn_te, batch_size=256)

model = dm.build_cnn()
dm.train_loop(model, mn_tr_loader, epochs=3, lr=1e-3, device=DEVICE, val_loader=mn_te_loader)
dm.save_model(model, str(PRE))
print("MNIST test acc      :", round(100*dm.accuracy(model, mn_te_loader, DEVICE), 2), "%")
pre = dm.evaluate(model, test_cells, DEVICE)
print("MNIST-only on OURS  :", round(100*pre["overall"], 2), "%   (domain gap)")

## 2. Fine-tune on our digit cells

Start from the MNIST weights, 90/10 train/val split of the 80-form digits, augment, save best-by-val-acc, early-stop. Then evaluate on the **20 held-out forms**.

In [ ]:
rng = random.Random(42); shuf = train_cells[:]; rng.shuffle(shuf)
n_val = max(1, int(len(shuf) * 0.1))
ft_val, ft_train = shuf[:n_val], shuf[n_val:]
tr_loader = DataLoader(dm._make_dataset(ft_train, augment=True),  batch_size=128, shuffle=True)
va_loader = DataLoader(dm._make_dataset(ft_val,   augment=False), batch_size=256)

model = dm.load_model(str(PRE), DEVICE)            # init from MNIST
hist = dm.train_loop(model, tr_loader, epochs=30, lr=5e-4, device=DEVICE,
                     val_loader=va_loader, ckpt_path=str(CKPT), patience=6)

best = dm.load_model(str(CKPT), DEVICE)            # best-by-val-acc
res = dm.evaluate(best, test_cells, DEVICE)
print(f"\nDIGIT CNN on 20 held-out forms: {100*res['overall']:.2f}% overall")
print("per-class %:", {k: (round(100*v,1) if v is not None else None)
                        for k,v in res['per_class'].items()})

## 3. From-scratch baseline (did MNIST help?)

Same data, no MNIST init — tells you whether pretraining mattered for your data size.

In [ ]:
scratch = dm.build_cnn()
dm.train_loop(scratch, tr_loader, epochs=30, lr=5e-4, device=DEVICE,
              val_loader=va_loader, patience=6)
sres = dm.evaluate(scratch, test_cells, DEVICE)
print(f"from-scratch on 20 forms : {100*sres['overall']:.2f}%")
print(f"MNIST-pretrained         : {100*res['overall']:.2f}%")

## 4. Downstream: full-field exact match on the 20 held-out forms

Assemble each digit comb field from the CNN's per-cell predictions and score exact match vs labels — compared to L1 where available. (insurance excluded here: its 3 letters need EasyOCR.)

In [ ]:
DIGIT_FIELDS = ["date_of_birth","date_of_visit","ssn","phone_number",
                "emergency_contact_phone","age"]
def norm(s): return " ".join(str(s).strip().split()).lower()
gt = labels.set_index("patient_id")

rows = {}
for f in DIGIT_FIELDS:
    cfg = FIELD_MAP["fields"][f]
    hit = sum(norm(dm.predict_comb_digits(best, str(CROPS / pid), f, cfg, DEVICE))
              == norm(gt.loc[pid, f]) for pid in val_pids)
    rows[f] = {"CNN_%": round(100*hit/len(val_pids), 1)}

PRED_L1 = ROOT / "data" / "generated" / "predictions_l1.csv"
if PRED_L1.exists():
    l1 = pd.read_csv(PRED_L1, dtype=str).fillna("").set_index("patient_id")
    for f in DIGIT_FIELDS:
        rows[f]["L1_%"] = round(100*sum(norm(l1.loc[p,f])==norm(gt.loc[p,f])
                                 for p in val_pids if p in l1.index)/len(val_pids), 1)

cmp = pd.DataFrame(rows).T
if "L1_%" in cmp: cmp["delta"] = cmp["CNN_%"] - cmp["L1_%"]
print(cmp.to_string())

## 5. Fine-tune curve

In [ ]:
import matplotlib.pyplot as plt
ep = [h["epoch"] for h in hist if h["val_acc"] is not None]
va = [100*h["val_acc"] for h in hist if h["val_acc"] is not None]
plt.figure(figsize=(7,4))
plt.plot(ep, va, "-o", ms=3)
plt.xlabel("epoch"); plt.ylabel("val accuracy %"); plt.title("Digit CNN fine-tune")
plt.grid(alpha=0.3); plt.tight_layout(); plt.show()